In [ ]:
# Import necessary libraries
import sympy as sp
from sympy import symbols, Matrix, sin, cos, diff, simplify
from sympy.physics.mechanics import dynamicsymbols

In [ ]:
# Define Variables
# Define time
t = symbols('t')

# Define Generalized Coordinates (as functions of time)
theta_n = dynamicsymbols('theta_n')  # Nacelle tower tilt
theta_s = dynamicsymbols('theta_s')  # Shaft tilt
psi = dynamicsymbols('psi')          # Blade spinning

# Define Generalized Velocities (first derivatives)
theta_n_dot = diff(theta_n, t)
theta_s_dot = diff(theta_s, t)
psi_dot = diff(psi, t)

# Define Generalized Accelerations (second derivatives)
theta_n_ddot = diff(theta_n_dot, t)
theta_s_ddot = diff(theta_s_dot, t)
psi_ddot = diff(psi_dot, t)

# Lists for coordinates and velocities
q = Matrix([theta_n, theta_s, psi])
q_dot = Matrix([theta_n_dot, theta_s_dot, psi_dot])
q_ddot = Matrix([theta_n_ddot, theta_s_ddot, psi_ddot])

# Define Constants
m_n, m_s, m_b = symbols('m_n m_s m_b')  # Mass of Nacelle, Mass of Blade
g = symbols('g')  # Gravity

# Local COM positions (in their own body frames)
X_n, Y_n, Z_n = symbols('X_n Y_n Z_n')
X_s, Y_s, Z_s = symbols('X_s Y_s Z_s')
X_b, Y_b, Z_b = symbols('X_b Y_b Z_b')

# Diagonal Inertia Tensors (in their own body frames)
I_n_xx, I_n_yy, I_n_zz = symbols('I_n_xx I_n_yy I_n_zz')
I_s_xx, I_s_yy, I_s_zz = symbols('I_s_xx I_s_yy I_s_zz')
I_b_xx, I_b_yy, I_b_zz = symbols('I_b_xx I_b_yy I_b_zz')

In [ ]:
# KINEMATICS

# Frame Rotation Matrices
# R_n: Nacelle Frame (n) relative to Tower (t)
tRn = Matrix([
    [cos(theta_n), 0, sin(theta_n)],
    [0, 1, 0],
    [-sin(theta_n), 0, cos(theta_n)]
])

# R_n_s: Shaft Frame (s) relative to Nacelle (n)
nRs = Matrix([
    [cos(theta_s), 0, sin(theta_s)],
    [0, 1, 0],
    [-sin(theta_s), 0, cos(theta_s)]
])

# R_s_b: Blade Frame (b) relative to Shaft (s)
sRb = Matrix([
    [1, 0, 0],
    [0, cos(psi), sin(psi)],
    [0, -sin(psi), cos(psi)]
])

# Combined Rotation Matrices
# R_t_s: Shaft (s) to Tower (t)
tRs = tRn * nRs
# R_t_b: Blade (b) to Tower (t)
tRb = tRn * nRs * sRb

# Angular Velocities (t frame)
# y_t_hat = [0, 1, 0], y_n_hat = tRn * [0, 1, 0] = [0, 1, 0]
# x_s_hat = tRs * [1, 0, 0]
y_n_hat = tRn[:, 1]  # Second column of tRn
x_b_hat = tRb[:, 0]  # First column of tRs

omega_n = theta_n_dot * y_n_hat
omega_s = (theta_n_dot + theta_s_dot) * y_n_hat
omega_b = omega_s + psi_dot * x_b_hat

# --- COM Positions (in inertial 't' frame) ---
r_n_local = Matrix([X_n, Y_n, Z_n])  # Nacelle COM in n-frame
r_b_local = Matrix([X_b, Y_b, Z_b])  # Blade COM in b-frame

r_G_n = tRn * r_n_local
r_G_b = tRb * r_b_local

# --- Linear Velocities (in inertial 't' frame) ---
# Differentiate the position vectors w.r.t. time
v_G_n = diff(r_G_n, t)
v_G_b = diff(r_G_b, t)